### Notebook to process and manipulate coverage geopackage
- Works for gpkg that have been created with `01_Create_cov.sh`
- Iterates over user defined directory path and processes all geopackages inside whose filenames end with "_Area.gpkg"
- *TODO:* Define and add metadata; add logic to append metadata
- *TODO:* Add Osis link -> access API to match Osis link to correct cruise

In [2]:
import shutil
import geopandas as gpd
import pandas as pd
import os
import numpy as np
from pathlib import Path
import glob

#### 3. Process and add metadata to coverage polygon
- remove zero value box around track
- dissolve fields if multiple are present
- calculate area of swath coverage
- add name of original raster
- **⚡ Change path name to the folder where the geopackage is located in**
- **⚡ TODO: add OSIS Link as metadata**

In [1]:
gpkg_path = "/Users/mschumacher/Docs_Data/Bathy/Bathy_Workflow/test/SO3000/SO3000_products/_cov" # folder that contains coverage geopackages that shall be processed

In [ ]:
for gpkg in glob.glob(f"{gpkg_path}/*_Area.gpkg"):
    gpkg_df = gpd.read_file(gpkg, index_col = False)
    gpkg_df_red = gpkg_df.drop(gpkg_df[gpkg_df['DN'] == 0].index)
    gpkg_diss = gpkg_df_red.dissolve(by = 'DN')
    base_grid_filename = os.path.basename(gpkg).replace("_Area", "")
    gpkg_diss['Filename'] = base_grid_filename
    gpkg_diss['Area [km2]'] = np.sum(gpkg_diss['geometry'].area)/(1000*1000)
    out_gpkg = gpkg.replace("Area", "Coverage")
    print(out_gpkg)
    gpkg_diss.to_file(out_gpkg, driver='GPKG', mode='w')

/Users/mschumacher/Docs_Data/VS/miniconda3/envs/.blueheart/lib/python3.12/site-packages/pyogrio/raw.py:198: RuntimeWarning: driver GPKG does not support open option INDEX_COL
  return ogr_read(


### ----------- 4. Copy processed geopackages to respective _cov folder on filer --------------------
- **⚡ If it is just a handful of files, this can also bve done manually!**
- Careful to copy the right file to its correct subfolder by comparing cruise names of parent dir and cruise name that is contained in filename
- **If names are misspelled, the copying won't work. Check along for spelling errors with the missing file lists.**
- **⚡ Note that you need to change platform name and maybe sometimes paths to directories**


In [ ]:
# Set vessel name for python
platform = "MERIAN"  # "MERIAN", "METEOR", "SONNE"

In [ ]:
# Copy gpkgs to _cov folder on filer
# Change source and dest paths accordingly

gpkg_path = f"/Users/mschumacher/Docs_Data/Bathy/Processing/{platform}"
base_dir = f"/Volumes/bathymetry/_blueheart/00_{platform}/{platform}_GEOMAR"
os.chdir(base_dir)

# Create list from _cov dir paths
dst_cov = glob.glob("*/_cov", recursive=True)
dst_cov += glob.glob("*/*/_cov", recursive=True)
cov_cruise = [cov.split("/")[0] for cov in dst_cov]

dst_cov = [ os.path.abspath(cov) for cov in dst_cov ]

#print(grid_files)

for gpkg in glob.glob(f"{gpkg_path}/*_Coverage.gpkg"):
    for cruise, dst in zip(cov_cruise, dst_cov):
        gpkg_cruise = os.path.basename(gpkg).split("_")[0]  # get cruise number from geopackage filename
        if gpkg_cruise == cruise:
            dst_name = os.path.join(dst, os.path.basename(gpkg))
            print(f"Copying {gpkg} to {dst_name}")
            shutil.copyfile(gpkg, dst_name)